This is Model One BM25 from the WANDS TSV file , we have three datasets namley 
1)Product df , which is the product catalogue of what the store has and what can be search for by customer.42994 products 
2)The second dataset is the Query df of what customers have actually searched for 480 queries 
And than the final dataset is the 3)Label df of human relevance search results and the key evaluation that our model will be evaluated on 233 448 human labeled query and product relevance pairs with the label being , exact , partial or irrelevant match.

Data preprocessing techniques  BM25 a non supervised  lexical retrieval model ( how may words overlap the query and the product.),  BM25 scores and ranks documents according to their relevance to a query based on lexical metching . I am testing this model as a information retrieval and ranking task and towards the end of the notebook I expirement with finding products and seeing how they rank and using the label df as guide.  




In [39]:
import pandas as pd
import numpy as np
import re

# Load dataset files
query_df = pd.read_csv('query.csv', sep='\t')
product_df = pd.read_csv('product.csv.zip', sep='\t')
label_df = pd.read_csv('label.csv.zip', sep='\t')

# Check shape to confirm successful load
print(f"Query shape: {query_df.shape}")
print(f"Products shape: {product_df.shape}")
print(f"Labels shape: {label_df.shape}")

Query shape: (480, 3)
Products shape: (42994, 9)
Labels shape: (233448, 4)


In [40]:
# Check missing values across all dataframes
dfs = {'Query': query_df, 'Product': product_df, 'Label': label_df}

for name, df in dfs.items():
    print(f"\nMissing values in {name}:")
    print(df.isna().sum())


Missing values in Query:
query_id       0
query          0
query_class    6
dtype: int64

Missing values in Product:
product_id                0
product_name              0
product_class          2852
category hierarchy     1556
product_description    6008
product_features          0
rating_count           9452
average_rating         9452
review_count           9452
dtype: int64

Missing values in Label:
id            0
query_id      0
product_id    0
label         0
dtype: int64


In [41]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

# Clean product text fields
cols_to_clean = ['product_name', 'product_class', 'category hierarchy', 'product_description', 'product_features']

for col in cols_to_clean:
    product_df[col] = product_df[col].fillna('')
    product_df[col + '_clean'] = product_df[col].apply(clean_text)

In [42]:
# Clean product text fields
cols_to_clean = ['product_name', 'product_class', 'category hierarchy', 'product_description', 'product_features']

for col in cols_to_clean:
    product_df[col] = product_df[col].fillna('')
    product_df[col + '_clean'] = product_df[col].apply(clean_text)

In [43]:
# Combine metadata into single search document
product_df['search_document'] = (
    product_df['product_name_clean'] + " " + 
    product_df['product_class_clean'] + " " + 
    product_df['category hierarchy_clean'] + " " + 
    product_df['product_description_clean'] + " " + 
    product_df['product_features_clean']
)

In [44]:
product_df[['product_name', 'search_document']].head()

,product_name,search_document
0,solid wood platform bed,solid wood platform bed beds furniture bedroo...
1,all-clad 7 qt . slow cooker,allclad 7 qt slow cooker slow cookers kitchen...
2,all-clad electrics 6.5 qt . slow cooker,allclad electrics 65 qt slow cooker slow cook...
3,all-clad all professional tools pizza cutter,allclad all professional tools pizza cutter sl...
4,baldwin prestige alcott passage knob with roun...,baldwin prestige alcott passage knob with roun...


In [45]:
from rank_bm25 import BM25Okapi

# Tokenize documents for BM25 indexing
tokenized_corpus = [doc.split() for doc in product_df['search_document']]

# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

In [46]:
# Print sample tokenized document
print("Sample tokenized document:")
print(tokenized_corpus[0])

Sample tokenized document:
['solid', 'wood', 'platform', 'bed', 'beds', 'furniture', 'bedroom', 'furniture', 'beds', 'headboards', 'beds', 'twin', 'beds', 'good', 'deep', 'sleep', 'can', 'be', 'quite', 'difficult', 'to', 'have', 'in', 'this', 'busy', 'age', 'fortunately', 'there', 's', 'an', 'antidote', 'to', 'such', 'a', 'problem', 'a', 'nice', 'quality', 'bed', 'frame', 'like', 'the', 'acacia', 'kaylin', 'solidly', 'constructed', 'from', 'acacia', 'wood', 'this', 'bed', 'frame', 'will', 'stand', 'the', 'test', 'of', 'time', 'and', 'is', 'fit', 'to', 'rest', 'your', 'shoulders', 'on', 'for', 'years', 'and', 'years', 'its', 'sleek', 'natural', 'wood', 'grain', 'appearance', 'provides', 'a', 'pleasant', 'aesthetic', 'to', 'adorn', 'any', 'bedroom', 'acting', 'both', 'as', 'a', 'decorative', 'piece', 'as', 'well', 'as', 'a', 'place', 'to', 'give', 'comfort', 'after', 'a', 'hard', 'day', 'of', 'work', 'our', 'bed', 'frame', 'is', 'designed', 'to', 'give', 'ample', 'underbed', 'space', 'fo

In [47]:
# Index corpus
bm25 = BM25Okapi(tokenized_corpus)

In [48]:
# Test search with sample query
test_query = "salon chair"
tokenized_query = clean_text(test_query).split()

# Get top 5 matching product documents
top_docs = bm25.get_top_n(tokenized_query, product_df['search_document'].tolist(), n=5)

for i, doc in enumerate(top_docs, 1):
    print(f"{i}. {doc}")

1. massage chair massage chairs furniture  living room furniture  chairs  seating  massage chairs features heavy duty steel frame  premium chrome round base  heavy duty hydraulic pump make height adjustment easier  doublereinforced saddle sticking prevents ripping and tearing  small cell high density foam for extra comfort of your client  easy to assemble and clean  barber chair salon chair styling chair salon chairs hair styling chair haircut chair salon chair haircut chair salon barber chair heavy duty barber chair shampoo chair hydraulic salon chair salon stool chair beauty hair chair shampoo chair adjustable swivel barber chairs shampoo chair with leg rest salon stool for hair stylist stool salon haircut chair home hydraulic lift styling chair heavy duty salon chair overallheighttoptobottom355productcare  dry clothupholsterymaterial  faux leathercolor  blackseatdepthfronttoback195seatwidthsidetoside195recliningtype  manualoveralldepthfronttoback23removablecushionlocation  backweigh

In [49]:
# 1. Define query and compute scores
test_query = "salon chair"
tokenized_query = clean_text(test_query).split()
bm25_scores = bm25.get_scores(tokenized_query)

# 2. Get top 10 scoring product indices
top_10_idx = np.argsort(bm25_scores)[-10:][::-1]

# 3. Build results table
cols = ['product_id', 'product_name', 'product_class', 'category hierarchy']
bm25_results = product_df.iloc[top_10_idx][cols].copy()

bm25_results['bm25_score'] = bm25_scores[top_10_idx]
bm25_results.insert(0, 'rank', range(1, 11))

bm25_results.reset_index(drop=True)

,rank,product_id,product_name,product_class,category hierarchy,bm25_score
0,1,15612,massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,16.914649
1,2,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,15.212085
2,3,7466,reclining massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.128911
3,4,7467,reclining faux leather massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.090445
4,5,7465,hair salon chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.089227
5,6,22462,hard floor straight round chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.543773
6,7,22463,hard floor straight rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.340485
7,8,25431,barberpub salon massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,13.199099
8,9,36889,beveled rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats,12.946083
9,10,29744,genuine leather reclining adjustable width mas...,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,12.831897


In [50]:
# Look up query ID and ground truth labels
query_id = query_df.loc[query_df['query'] == 'salon chair', 'query_id'].iloc[0]
ground_truth = label_df[label_df['query_id'] == query_id][['product_id', 'label']]

# Benchmark BM25 results against human judgments
bm25_evaluation = bm25_results.merge(ground_truth, on='product_id', how='left')
bm25_evaluation['label'] = bm25_evaluation['label'].fillna('Unjudged')

# Display benchmark table
print(f"Query ID: {query_id}")
bm25_evaluation

Query ID: 0


,rank,product_id,product_name,product_class,category hierarchy,bm25_score,label
0,1,15612,massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,16.914649,Exact
1,2,7468,mercer41 hair salon chair hydraulic styling ch...,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,15.212085,Exact
2,3,7466,reclining massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.128911,Exact
3,4,7467,reclining faux leather massage chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.090445,Exact
4,5,7465,hair salon chair,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,15.089227,Exact
5,6,22462,hard floor straight round chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.543773,Irrelevant
6,7,22463,hard floor straight rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats / Ha...,13.340485,Irrelevant
7,8,25431,barberpub salon massage chair,Massage Chairs,Furniture / Living Room Furniture / Chairs & S...,13.199099,Exact
8,9,36889,beveled rectangular chair mat,Chairmats,Furniture / Office Furniture / Chair Mats,12.946083,Irrelevant
9,10,29744,genuine leather reclining adjustable width mas...,Massage Chairs|Recliners,Furniture / Living Room Furniture / Chairs & S...,12.831897,Exact


In [51]:
import numpy as np

# Map labels to numeric relevance scores
label_map = {'Exact': 2, 'Partial': 1, 'Irrelevant': 0, 'Unjudged': 0}
relevances = bm25_evaluation['label'].map(label_map).fillna(0).values

# 1. Precision@10
relevant_retrieved = np.sum(relevances > 0)
precision_at_10 = relevant_retrieved / len(relevances)

# 2. Recall@10 (against all known relevant items in ground truth)
total_relevant_in_gt = len(ground_truth[ground_truth['label'].isin(['Exact', 'Partial'])])
recall_at_10 = relevant_retrieved / total_relevant_in_gt if total_relevant_in_gt > 0 else 0.0

# 3. NDCG@10 Calculation
def compute_dcg(rel):
    discounts = np.log2(np.arange(len(rel)) + 2)
    return np.sum((2**rel - 1) / discounts)

dcg = compute_dcg(relevances)
idcg = compute_dcg(np.sort(relevances)[::-1])
ndcg_at_10 = dcg / idcg if idcg > 0 else 0.0

# Print results summary
print(f"Precision@10 : {precision_at_10:.4f}")
print(f"Recall@10    : {recall_at_10:.4f}")
print(f"NDCG@10      : {ndcg_at_10:.4f}")

Precision@10 : 0.7000
Recall@10    : 0.0023
NDCG@10      : 0.9766


In [52]:
# Mark 'Exact' and 'Partial' labels as relevant
bm25_evaluation['is_relevant'] = bm25_evaluation['label'].isin(['Exact', 'Partial'])

# 1. Precision@10
relevant_in_top10 = bm25_evaluation['is_relevant'].sum()
precision_at_10 = relevant_in_top10 / 10

# 2. Recall@10
total_relevant_in_gt = len(ground_truth[ground_truth['label'].isin(['Exact', 'Partial'])])
recall_at_10 = relevant_in_top10 / total_relevant_in_gt if total_relevant_in_gt > 0 else 0.0

# 3. NDCG@10
rel_scores = bm25_evaluation['label'].map({'Exact': 2, 'Partial': 1, 'Irrelevant': 0, 'Unjudged': 0}).fillna(0).values
discounts = np.log2(np.arange(10) + 2)

dcg = np.sum((2**rel_scores - 1) / discounts)
idcg = np.sum((2**np.sort(rel_scores)[::-1] - 1) / discounts)
ndcg_at_10 = dcg / idcg if idcg > 0 else 0.0

# Output results
print(f"Query: {test_query}")
print(f"Relevant products in Top 10: {relevant_in_top10}")
print(f"Precision@10 : {precision_at_10:.3f}")
print(f"Recall@10    : {recall_at_10:.3f}")
print(f"NDCG@10      : {ndcg_at_10:.3f}")

Query: salon chair
Relevant products in Top 10: 7
Precision@10 : 0.700
Recall@10    : 0.002
NDCG@10      : 0.977
